# 1. Eksploracyjna analiza danych

### Wczytanie danych (i bibliotek)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve, mean_squared_error, 
                              r2_score, classification_report, confusion_matrix)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv('Hotel Reservations.csv')
print(f"Liczba wierszy: {df.shape[0]}, liczba kolumn: {df.shape[1]}")
df.head()

### Typy danych i zmiennych

In [ ]:
df.info()

In [ ]:
print("Zmienne numeryczne")
df.describe()

In [ ]:
print("Zmienne opisowe")
df.describe(include='object')

# TODO 
### zrozumienie zmiennych? rozklady cech, korelacje, wykresy itd

### Sprawdzenie brakujących danych

In [ ]:
print("Liczba brakujących wartości")
df.isnull().sum()

In [ ]:
print("Liczba duplikatów")
df.duplicated().sum()

In [ ]:
# df['booking_status'].value_counts(normalize=True) # Można tym sprawdzić sobie stosunek cancelled/not cancelled

# 2. Przetwarzanie danych

### Usunięcie kolumny Booking_ID

In [ ]:
df = df.drop('Booking_ID', axis = 1)
df.head()

### Kodowanie zmiennej docelowej

In [ ]:
target = {'Canceled': 1, 'Not_Canceled': 0} # Zmiana booking_status na liczbe
df['booking_status'] = df['booking_status'].replace(target)
print("Canceled = 1, Not_Canceled = 0")
df.describe()

### Podział danych na zbiór treningowy, walidacyjny i testowy w proporcjach 60/20/20

In [ ]:
X = df.drop('booking_status', axis = 1)
y = df['booking_status']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val) # 0.25 * 0.80 = 0.20

### Kodowanie zmiennych kategorycznych

In [ ]:
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeryczne cechy:", num_features)
print("Kategoryczne cechy:", cat_features)

In [ ]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder()) # handle_unknown='ignore', sparse_output=False
])
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

data_pipeline.fit(X_train)

feature_names = (
    pd.Index(data_pipeline.named_steps['preprocessor'].get_feature_names_out())
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

X_train_processed = pd.DataFrame(data_pipeline.transform(X_train),
                                columns=feature_names)

X_val_processed = pd.DataFrame(data_pipeline.transform(X_val),
                                columns=feature_names)

X_test_processed = pd.DataFrame(data_pipeline.transform(X_test),
                                columns=feature_names)

# 3. Trening i ewaluacja modeli

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_v, y_v, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)
    y_prob = model.predict_proba(X_v)[:, 1]
    return {
        'Model': name,
        'Accuracy':  np.round(accuracy_score(y_v, y_pred), 4),
        'Precision': np.round(precision_score(y_v, y_pred), 4),
        'Recall':    np.round(recall_score(y_v, y_pred), 4),
        'F1-Score':  np.round(f1_score(y_v, y_pred), 4),
        'ROC-AUC':   np.round(roc_auc_score(y_v, y_prob), 4),
    }

results = []
trained_models = {}

### Regresja logistyczna

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
res = evaluate_model(lr, X_train_processed, y_train, X_val_processed, y_val, 'Logistic Regression')
results.append(res)
trained_models['Logistic Regression'] = lr
print("Wyniki regresji liniowej na zbiorze walidacyjnym:\n")
for k, v in res.items():
    print(f"{k}: {v}")


Wyniki regresji liniowej na zbiorze walidacyjnym:
Model: Logistic Regression
Accuracy: 0.8065
Precision: 0.7372
Recall: 0.6361
F1-Score: 0.6829
ROC-AUC: 0.8655
